# Badge 4: Data Lake Workshop: Lesson 9: Lions & Tigers & Bears, Oh My!!!

External tables, the secure materialized view and an Iceberg table.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `BADGE4_DLKW_ADMIN` (ACCOUNTADMIN), `BADGE4_DLKW_ROLE` (SYSADMIN) and `BADGE4_DLKW_WH` (COMPUTE_WH).
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.
- Files the course has learners download are on `@UTIL_DB.PUBLIC.BADGE4_DOWNLOADS`, standing in for the learner's Downloads folder.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE4_DLKW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE4_DLKW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE4_DLKW_WH"      # stands in for the course's COMPUTE_WH


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
use schema MELS_SMOOTHIE_CHALLENGE_DB.TRAILS;
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: 📓 What Are All These Things?

Reading and quiz only.

## Page: 📓 What is a Materialized View?

Reading and quiz only.

## Page: 📓 What is an External Table?

Reading and quiz only (DLKW_147-151 are docs screenshots).

## Page: 🥋 Let's Try to Create an External Table

> ⚠️ **Audit note:** The course says this first attempt on an internal stage fails (b4_1.png). It runs as given and reports what happens.

In [ ]:
%%sql -r external_table_on_internal_stage
execute immediate 'begin
  execute immediate ''create or replace external table T_CHERRY_CREEK_TRAIL(
my_filename varchar(100) as (metadata$filename::varchar(100))
)
location= @trails_parquet
auto_refresh = true
file_format = (type = parquet)'';
  return ''ran without error'';
exception when other then
  return ''Error (as the course predicts): '' || sqlerrm;
end;';

## Page: 🥋 Let's Try That External Table, Again!

> **Course:** Create an External Stage for an External Table (1-dlkw, 2-dlkw, 3-dlkw): Catalog → MELS_SMOOTHIE_CHALLENGE_DB → TRAILS → Create → Stage → Amazon S3; Stage Name **EXTERNAL_AWS_DLKW**; URL **s3://uni-dlkw**; no authentication.

In [ ]:
%%sql -r create_external_stage
-- SQL equivalent of the Create Stage dialog (2-dlkw)
use role {{ADMIN_ROLE}};
use warehouse {{WAREHOUSE}};
use schema MELS_SMOOTHIE_CHALLENGE_DB.TRAILS;
create stage EXTERNAL_AWS_DLKW
  url = 's3://uni-dlkw';
list @external_aws_dlkw;

> **Course:** Change the length of the file name and change the name of the STAGE to your new External Stage, then run the command again (4-dlkw). After creating the table, run a SELECT * (5-dlkw).

In [ ]:
%%sql -r create_external_table
-- From screenshot 4-dlkw
create or replace external table T_CHERRY_CREEK_TRAIL(
   my_file_name varchar(100) as (metadata$filename::varchar(100))
)
location= @external_aws_dlkw
auto_refresh = true
file_format = (type = parquet);

In [ ]:
%%sql -r select_external_table
select * from t_cherry_creek_trail;

## Page: 🥋 Let's Try that Materialized View Again...

In [ ]:
%%sql -r create_secure_mv
create secure materialized view MELS_SMOOTHIE_CHALLENGE_DB.TRAILS.SMV_CHERRY_CREEK_TRAIL(
	POINT_ID,
	TRAIL_NAME,
	LNG,
	LAT,
	COORD_PAIR,
    DISTANCE_TO_MELANIES
) as
select 
 value:sequence_1 as point_id,
 value:trail_name::varchar as trail_name,
 value:latitude::number(11,8) as lng,
 value:longitude::number(11,8) as lat,
 lng||' '||lat as coord_pair,
 locations.distance_to_mc(st_makepoint(lng, lat)) as distance_to_melanies
from t_cherry_creek_trail;

In [ ]:
%%sql -r select_secure_mv
-- 8-dlkw: the materialized view over the external table
select * from MELS_SMOOTHIE_CHALLENGE_DB.TRAILS.SMV_CHERRY_CREEK_TRAIL;

## Page: 🤖 DORA DLKW09

### 🤖 DORA check DLKW09 goes here
The course has the learner run **DLKW09** (secure materialized view has 3,526 rows) at this point.
It is in `dora/dora_tests.ipynb`, section **DLKW09**, as a local check plus the exact DORA code.

## Page: 📓 What is an Apache Iceberg Table?

Reading and quiz only.

## Page: 🥋 Set Up Your Iceberg External Volume and User

> **Course:** For the remainder of this workshop, you should do your work using the ACCOUNTADMIN Snowflake role.

In [ ]:
%%sql -r create_external_volume
-- Account-level object: runs as the real ACCOUNTADMIN
use role accountadmin;
CREATE OR REPLACE EXTERNAL VOLUME iceberg_external_volume
   STORAGE_LOCATIONS =
      (
         (
            NAME = 'iceberg-s3-us-west-2'
            STORAGE_PROVIDER = 'S3'
            STORAGE_BASE_URL = 's3://uni-dlkw-iceberg'
            STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::321463406630:role/dlkw_iceberg_role'
            STORAGE_AWS_EXTERNAL_ID = 'dlkw_iceberg_id'
         )
      );

In [ ]:
%%sql -r describe_external_volume
DESC EXTERNAL VOLUME iceberg_external_volume;

> ⚠️ **Audit note:** The course now has the learner enter the volume's `STORAGE_AWS_IAM_USER_ARN` in the YSA badge-management app (DLKW_170). That adds the user to the AWS policy of the course's bucket; it cannot be done from this account, so the Iceberg table below may fail with an access error until it is done.

## Page: 🥋 Set Up Your Apache Iceberg DB and Table

In [ ]:
%%sql -r create_iceberg_db
use role accountadmin;
create database my_iceberg_db
  catalog = 'SNOWFLAKE'
  external_volume = 'iceberg_external_volume';

In [ ]:
%%sql -r iceberg_table_name
use role accountadmin;
use database MY_ICEBERG_DB;
use schema PUBLIC;
use warehouse {{WAREHOUSE}};
set table_name = 'CCT_'||current_account();
select $table_name;

> ⚠️ **Audit note:** Verified 2026-11-14: creating the table fails with `Error assuming AWS_ROLE ... is not authorized to perform: sts:AssumeRole on resource: arn:aws:iam::321463406630:role/dlkw_iceberg_role`. The course's AWS role trusts only the volume IAM users that Snowflake Education adds through the YSA app (DLKW_170), which only a learner's course account can do. The next three cells run as given and report the result, so DLKW10 cannot pass here.

In [ ]:
%%sql -r create_iceberg_table
use role accountadmin;
use database MY_ICEBERG_DB;
use schema PUBLIC;
use warehouse {{WAREHOUSE}};
execute immediate 'begin
  execute immediate ''set table_name = ''''CCT_''''||current_account();

create iceberg table identifier($table_name) (
    point_id number(10,0),
    trail_name string,
    coord_pair string,
    distance_to_melanies decimal(20,10),
    user_name string
)
  base_location = $table_name
  as
  select top 100
      point_id,
      trail_name,
      coord_pair,
      distance_to_melanies,
      current_user()
  from MELS_SMOOTHIE_CHALLENGE_DB.TRAILS.SMV_CHERRY_CREEK_TRAIL'';
  return ''ran without error'';
exception when other then
  return ''Error (as the course predicts): '' || sqlerrm;
end;';

In [ ]:
%%sql -r select_iceberg_table
use role accountadmin;
use database MY_ICEBERG_DB;
use schema PUBLIC;
execute immediate 'begin
  execute immediate ''select * from identifier($table_name)'';
  return ''ran without error'';
exception when other then
  return ''Error (as the course predicts): '' || sqlerrm;
end;';

## Page: 📓 What's the Big Deal Here?

In [ ]:
%%sql -r update_iceberg_table
use role accountadmin;
use database MY_ICEBERG_DB;
use schema PUBLIC;
execute immediate 'begin
  execute immediate ''update identifier($table_name)
set user_name = ''''I am amazing!!''''
where point_id = 1'';
  return ''ran without error'';
exception when other then
  return ''Error (as the course predicts): '' || sqlerrm;
end;';